# Izdelava aplikacij za generiranje slik (OpenAI)

Veliko več kot samo generiranje besedila lahko opravijo veliki jezikovni modeli (LLM). Prav tako lahko generirate slike iz besedilnih opisov. Slike kot modaliteta so uporabne v medicinski tehnologiji, arhitekturi, turizmu, razvoju iger, marketingu in še več. V tej lekciji delamo z današnjimi modeli **GPT Image** na platformi OpenAI.

## Cilji učenja

Do konca te lekcije boste sposobni:

- Pojasniti, kaj je generiranje slik in kje je uporabno.
- Razumeti družino modelov `gpt-image` in kako se razlikujejo od starih modelov DALL·E.
- Izdelati aplikacijo za generiranje slik in urejati slike.

## Kaj je generiranje slik?

Modeli za generiranje slik ustvarjajo slike iz besedilnega poziva. Sodobni modeli, kot je `gpt-image`, med učenjem spoznajo povezavo med besedilom in slikami, nato pa postopoma spremenijo naključni šum v sliko, ki ustreza vašemu pozivu.

Dve znani družini modelov slik sta:

- **`gpt-image` (OpenAI)** — trenutna generacija, uporabljena v tej lekciji. Podpira generiranje slike iz besedila in urejanje slike (vdelava s masko).
- **Midjourney** — priljubljen model tretje osebe s svojo storitvijo in delovnim procesom na Discordu.

> Starejši modeli slik OpenAI — **DALL·E 2** in **DALL·E 3** — so zastareli. DALL·E 3 ni več na voljo za nove implementacije, funkcije, kot je `create_variation`, pa so obstajale samo v DALL·E 2. Za nove aplikacije uporabljajte modele `gpt-image`.

> **Pomembno:** modeli `gpt-image` vrnejo ustvarjeno sliko kot **base64** (`b64_json`), ne kot URL. Vaša koda dekodira niz base64 v bajte in jih shrani — ne obstaja URL slike za prenos.


## Izdelava vaše prve aplikacije za ustvarjanje slik

Kaj torej potrebujete za izdelavo aplikacije za ustvarjanje slik? Potrebujete naslednje knjižnice:

- **python-dotenv**, zelo priporočamo uporabo te knjižnice za shranjevanje vaših skrivnosti v datoteko *.env* ločeno od kode.
- **openai**, ta knjižnica se uporablja za interakcijo z OpenAI API.
- **pillow**, za delo s slikami v Pythonu.


1. Ustvarite datoteko *.env* z naslednjo vsebino:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Zberite zgornje knjižnice v datoteko imenovano *requirements.txt* tako:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Nato ustvarite virtualno okolje in namestite knjižnice:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Za Windows uporabite naslednje ukaze za ustvarjanje in aktivacijo vašega virtualnega okolja:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodajte naslednjo kodo v datoteko z imenom *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Ustvari OpenAI objekt (prebere OPENAI_API_KEY iz vaše .env datoteke)
    client = openai.OpenAI()


    try:
        # Ustvari sliko z uporabo API-ja za generiranje slik
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Vnesite svoj besedilni poziv tukaj
            size='1024x1024',
            n=1
        )
        # Nastavi mapo za shranjeno sliko
        image_dir = os.path.join(os.curdir, 'images')

        # Če mapa ne obstaja, jo ustvari
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializiraj pot slike (upoštevaj, da mora biti datotečna končnica png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modeli vrnejo sliko kot base64 (b64_json), ne kot URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Prikaži sliko v privzetem pregledovalniku slik
        image = Image.open(image_path)
        image.show()

    # ujemi izjeme
    except openai.BadRequestError as err:
        print(err)

    ```

Razložimo to kodo:

- Najprej uvozimo knjižnice, ki jih potrebujemo, vključno z OpenAI knjižnico, dotenv, modul base64 in knjižnico Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Nato ustvarimo odjemalca, ki prebere API ključ iz vaše datoteke ``.env``.

    ```python
    # Ustvari OpenAI objekt
    client = openai.OpenAI()
    ```

- Naprej generiramo sliko:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Vnesite besedilo poziva tukaj
        size='1024x1024',
        n=1
    )
    ```

    Modeli `gpt-image` vrnejo sliko kot niz **base64** v `data[0].b64_json`. Dekodiramo ga v bajte in zapišemo v datoteko — ni URL-ja za prenos.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na koncu odprejemo sliko in jo prikažemo s standardnim pregledovalnikom slik:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Več podrobnosti o generiranju slike

Oglejmo si parametre funkcije `images.generate`:

- **model**, je model slike, ki ga želite uporabiti (na primer `gpt-image-1`).
- **prompt**, je besedilni opis, ki se uporabi za generiranje slike. Tukaj je "Zajček na konju, drži liziko, na megleni travi, kjer rastejo trobentice".
- **size**, je velikost generirane slike (na primer `1024x1024`, `1536x1024`, `1024x1536` ali `"auto"`).
- **n**, je število generiranih slik. Tukaj generiramo eno.

> Modeli za slike ne uporabljajo parametra `temperature` — to je kontrola za generiranje besedila. Za večjo raznolikost ponovno pokličite API; za manjšo raznolikost pa naredite opis bolj specifičen.

## Dodatne zmožnosti generiranja slik

Videli ste, kako z nekaj vrsticami Pythona generirati sliko. Modeli `gpt-image` lahko tudi **urejajo** obstoječo sliko. Z zagotovljenim posnetkom, dodatno **masko** (ki označuje območje za spremembo) in opisom lahko spremenite del slike — na primer, dodate klobuk našemu zajčku.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# urejanja so prav tako vrnjena kot base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Osnovna slika vsebuje le zajca; končna slika ima klobuk na zajcu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
